# QueryChat Interaction Log Notebook

This notebook explains why the interaction log exists, walks through a full data pipeline, and illustrates each step with concrete examples drawn from the EPL Performance Dashboard.

**Log columns**
| Column | Description |
|---|---|
| `Timestamp` | UTC time the query was submitted |
| `Query` | Natural-language text the user typed |
| `Response` | Summary string — row count + column list returned by the AI filter |

## Why We Need the Log

The **AI Explorer** tab lets users ask free-text questions such as:

> *"show Liverpool away wins"*  
> *"matches where the home team scored at least 4 goals"*  
> *"only draws in the 2021/22 season"*

Claude Haiku translates these into SQL, filters `df_all`, and renders charts.  
Without a log we have **zero visibility** into:

- What questions users actually ask (vs. what we designed for)
- Which queries silently return 0 rows (broken SQL generation)
- Which teams / seasons users care about most
- Whether AI filter usage grows or stalls after launch
- Edge-cases that should become new example prompts in the sidebar

The log is the **feedback loop** between user behaviour and product improvement.

```
User types query
      │
      ▼
QueryChat (Claude Haiku) → SQL → filters df_all
      │
      ▼
log_interaction(query, response_summary, timestamp)
      │
      ├─► Google Sheets  (shared, collaborative)
      └─► logs/querychat_log.csv  (local fallback)
              │
              ▼
         This notebook
              │   
              ▼           
        Usage analytics 
```

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from collections import Counter
import re as _re
print('Libraries loaded.')

## Load the Log

The app writes to `logs/querychat_log.csv` (or Google Sheets).  

In [ ]:
LOG_PATH = Path('logs/querychat_log.csv')

if LOG_PATH.exists():
    df_log = pd.read_csv(LOG_PATH, parse_dates=['Timestamp'])
    df_log.columns = df_log.columns.str.strip().str.capitalize()
    print(f'Loaded real log: {len(df_log)} rows from {LOG_PATH}')
else:
    df_log = pd.DataFrame(columns=['Timestamp', 'Query', 'Response'])
    print(f'No log file found at {LOG_PATH} — continuing with an empty log DataFrame.')

df_log['Timestamp'] = pd.to_datetime(df_log['Timestamp'], errors='coerce')
df_log.head()

### What questions users actually ask (vs. what we designed for)

### Which queries silently return 0 rows (broken SQL generation)

### Which teams / seasons users care about most